# AI_Biomed: Multimodal Biomedical Research Walkthrough

This notebook provides an interactive, end-to-end walkthrough of the reconstructed **AI_Biomed** framework for **Chronic Kidney Disease (CKD) risk stratification and drug nephrotoxicity prediction** using multimodal clinical feature fusion with **BioBERT** embeddings and **XGBoost**.

### Pipeline Overview
1. **Cohort Ingestion & Schema Inspection**: Loading the 3 core biomedical datasets.
2. **Clinical Preprocessing & Feature Selection**: Imputing missing values via KNN ($k=5$), Min-Max scaling, and selecting the top 37 clinical features via ANOVA F-statistic.
3. **Biomedical NLP Embedding**: Generating clinical patient vignettes and extracting 768-dim contextual embeddings via BioBERT v1.1.
4. **Multimodal Feature Fusion**: Concatenating structured clinical features with PCA-projected BioBERT representations.
5. **Model Training & Benchmark Verification**: Validating baseline models and XGBoost against historical targets.
6. **SHAP Model Explainability**: Quantifying physiological drivers of renal disease progression.
7. **Drug Nephrotoxicity Screening**: Evaluating intrinsic drug toxicity and multi-stage CKD progression.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ensure project root is on PATH
project_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from ai_biomed.config import PathConfig, PreprocessingConfig, BioBERTConfig, HistoricalBenchmarkTargets
from ai_biomed.data.loader import load_ckd_general_dataset, load_ckd_nephrotoxic_dataset, load_drug_labels_dataset
from ai_biomed.data.preprocessor import ClinicalPreprocessor, NephrotoxicDataPreprocessor
from ai_biomed.data.text_generator import generate_patient_clinical_narratives
from ai_biomed.nlp.biobert_embedder import BioBERTEmbedder
from ai_biomed.fusion.feature_fusion import MultimodalFusionEngine
from ai_biomed.models.baselines import BaselineModels
from ai_biomed.models.xgboost_classifier import XGBoostCKDClassifier
from ai_biomed.evaluation.metrics import compute_evaluation_metrics, format_metrics_table
from ai_biomed.explainability.shap_explainer import BiomedSHAPExplainer

print("AI_Biomed research environment initialized successfully.")

## 1. Ingest Cohorts & Inspect Clinical Data

We ingest the general CKD patient cohort ($N=1,659$, 54 columns), the nephrotoxicity cohort ($N=1,500$, 37 columns), and the FDA drug labels corpus ($N=1,436$).

In [ ]:
cfg_path = PathConfig()
df_ckd = load_ckd_general_dataset(path_config=cfg_path)
df_nephro = load_ckd_nephrotoxic_dataset(path_config=cfg_path)
df_drugs = load_drug_labels_dataset(path_config=cfg_path)

print(f"General CKD Cohort Shape:          {df_ckd.shape}")
print(f"Nephrotoxic Drug Exposure Shape:   {df_nephro.shape}")
print(f"Drug Labels Corpus Shape:          {df_drugs.shape}")

df_ckd[["Age", "Gender", "SerumCreatinine", "GFR", "SystolicBP", "Diagnosis"]].head()

## 2. Clinical Data Preprocessing & 37 Salient Features

We apply KNN Imputation ($k=5$), Min-Max scaling into $[0, 1]$, and ANOVA F-statistic feature selection to isolate exactly 37 clinical features.

In [ ]:
prep_cfg = PreprocessingConfig()
preprocessor = ClinicalPreprocessor(config=prep_cfg)
X_clin_scaled, y = preprocessor.transform_full(df_ckd, target_col="Diagnosis")

print(f"Selected clinical features count: {len(preprocessor.selected_features_)}")
print(f"Scaled Feature Matrix Shape: {X_clin_scaled.shape}")
print(f"Class Distribution: 0 (Non-CKD) = {sum(y == 0)}, 1 (CKD) = {sum(y == 1)}")

## 3. Contextual Biomedical Embeddings via BioBERT

BioBERT (`dmis-lab/biobert-v1.1`) extracts 768-dimensional mean-pooled contextual embeddings from clinical patient vignettes.

In [ ]:
bert_cfg = BioBERTConfig()
embedder = BioBERTEmbedder(config=bert_cfg, path_config=cfg_path)
cache_file = cfg_path.biobert_ckd_cache_path

if cache_file.exists():
    print(f"Loading precomputed BioBERT embeddings from {cache_file.name}...")
    loaded = np.load(cache_file)
    X_biobert = loaded if isinstance(loaded, np.ndarray) else np.asarray(loaded)
else:
    print("Generating narratives and extracting BioBERT embeddings...")
    narratives = generate_patient_clinical_narratives(df_ckd)
    X_biobert = embedder.encode_texts(narratives, cache_path=cache_file)

print(f"BioBERT Embedding Matrix Shape: {X_biobert.shape}")

## 4. Multimodal Feature Fusion

We project the 768-dim BioBERT embeddings to 16 principal components and concatenate them with the 37 clinical features, yielding a balanced 53-dimensional multimodal representation.

In [ ]:
fusion_engine = MultimodalFusionEngine(use_pca_projection=True, pca_components=16)
fused_bundle = fusion_engine.fuse(
    X_clinical=X_clin_scaled,
    X_biobert=X_biobert,
    y=y,
    clinical_feature_names=preprocessor.selected_features_
)

print(f"Fused Multimodal Shape: {fused_bundle.X_fused.shape} (37 clinical + 16 BioBERT PCA)")
print(f"Top 5 Features: {fused_bundle.feature_names[:5]}")
print(f"Last 5 Features: {fused_bundle.feature_names[-5:]}")

## 5. Model Training & Historical Performance Reproduction

We train Logistic Regression, Random Forest, and tuned XGBoost, validating the results against the historical benchmark targets.

In [ ]:
from sklearn.model_selection import train_test_split

X_tr, X_te, y_tr, y_te = train_test_split(
    fused_bundle.X_fused, y, test_size=0.2, random_state=42, stratify=y
)

baselines = BaselineModels(random_state=42)
baselines.fit_all(X_tr, y_tr)
baseline_preds = baselines.predict_all(X_te)
baseline_probs = baselines.predict_proba_all(X_te)

metrics_list = []
for name in baseline_preds:
    m = compute_evaluation_metrics(y_te, baseline_preds[name], baseline_probs[name], model_name=name)
    metrics_list.append(m)

# XGBoost Champion
xgb_mod = XGBoostCKDClassifier(n_estimators=300, max_depth=5, learning_rate=0.03, random_state=42)
xgb_mod.fit(X_tr, y_tr)
pred_xgb = xgb_mod.predict(X_te)
prob_xgb = xgb_mod.predict_proba(X_te)
metrics_xgb = compute_evaluation_metrics(y_te, pred_xgb, prob_xgb, model_name="XGBoost (Multimodal Champion)")
metrics_list.append(metrics_xgb)

table_df = format_metrics_table(metrics_list)
table_df

## 6. Model Explainability & SHAP Feature Attribution

We compute SHAP values using `BiomedSHAPExplainer` to identify the most critical clinical features driving CKD risk predictions.

In [ ]:
explainer = BiomedSHAPExplainer(xgb_mod, feature_names=fused_bundle.feature_names)
explainer.explain(X_te, max_evals=200)
shap_df = explainer.get_feature_importance_df(top_n=15)
shap_df

## 7. Drug Nephrotoxicity & Progression Modeling

We evaluate drug-induced kidney disease (DIKD) risk across 1,500 patient-medication encounters with Gentamicin, Cisplatin, Amphotericin-B, Tobramycin, Vancomycin, Aspirin, Ibuprofen, and Paracetamol.

In [ ]:
from scripts.run_nephrotoxicity_pipeline import run_nephrotoxicity_pipeline

metrics_nephro_df = run_nephrotoxicity_pipeline(path_config=cfg_path, verbose=False)
metrics_nephro_df

---
### Citation & Contact
**AI_Biomed Framework**: Multimodal BioBERT & Clinical Feature Fusion for CKD & Nephrotoxicity Stratification  
**Author**: Mannat Goyal (<mannatgoyal2810@gmail.com>)  
**GitHub**: [https://github.com/mannatgoyal/AI_Biomed](https://github.com/mannatgoyal/AI_Biomed)